# Tutorial 150 — Project types and intents: make your own

When you create a project in the desktop app, the first choice is the
**Project type**: *Code repository* or *Literature collection*. That choice
decides a lot without any code:

- which files a new project reads by default;
- how every answer is instructed (the system prompt);
- which **intents** exist, that is, the kinds of request QAChat recognizes
  (a question to answer, a whole document to read, a follow-up, small talk),
  and for each one how much conversation history to include and how to
  find evidence.

Each project type is one JSON file in `qachat/project_types/`. This tutorial
explains every part of such a file, shows how the router uses the intents,
and builds a new project type for meeting notes. Intents can also run a
**tool** (an exact count, a structured report); tools have their own
tutorial, 160.

```
your question
     │
     ▼
router model ── reads the intent descriptions of the project type
     │ chooses one intent
     ▼
intent ── history policy: how many earlier turns to include
     │    retrieval mode: search, read a whole document, reuse, or none
     ▼
answer model ── project and intent instructions + evidence
```

## 1. Prepare QAChat

Each tutorial is independently runnable. These cells locate the repository,
make the local package importable, and call `prepare_notebook()` to verify the
project `.venv` environment. If Jupyter asks you to select the printed `.venv`
kernel, select it and rerun the cell.

In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")

In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)

In [ ]:
import json

import qachat

## 2. The built-in project types

`load_project_types()` reads every JSON file in `qachat/project_types/` and
validates it. The file name is the type's ID (`code.json` is `code`), and
the result is what the desktop app lists under *Project type*.

In [ ]:
from qachat.project_type import PROJECT_TYPES_DIR, load_project_types

print(f"Folder: {PROJECT_TYPES_DIR}\n")
for kind, item in load_project_types().items():
    print(f"{kind:12} {item.display_name}")
    print(f"{'':12} reads {', '.join(item.extensions)}")
    print(f"{'':12} {len(item.intents)} intents\n")

## 3. Anatomy of a project type

Open `qachat/project_types/literature.json` next to this notebook. Its
top-level fields are:

| Field | What it does |
|---|---|
| `display_name` | The name shown under *Project type*. |
| `description` | One sentence about the sources; the router reads it. |
| `extensions`, `max_file_bytes` | Which files a new project reads by default, and the largest file size. |
| `retrieval_instruction` | The task given to the embedding model with each search query. |
| `system_prompt` | Instructions for every answer in this project type. |
| `answer_guidance` | An extra instruction the desktop app adds to each question (optional). |
| `capabilities.code_graph` | Build the code graph: `true`, `false`, or `"auto"`. |
| `history.recent_turns` | How many recent turns count as conversation history. |
| `default_intent` | The intent used when routing fails (default: the first). |
| `welcome_prompts`, `suggestions` | What an empty chat shows in the desktop app. |
| `intents` | The kinds of request this project type recognizes. |

In [ ]:
literature = json.loads(
    (PROJECT_TYPES_DIR / "literature.json").read_text(encoding="utf-8")
)
for key, value in literature.items():
    if key == "intents":
        print(f"{key}: {', '.join(value)}")
    else:
        print(f"{key}: {str(value)[:70]}")

## 4. Intents

An intent is one kind of request. The router model reads every intent's
`description` and picks one per question, so the description is the most
important line: say what belongs to the intent, give a few example
questions, and say what does **not** belong when two intents are close.

The other fields decide what happens once the intent is chosen:

| Field | Values | What it does |
|---|---|---|
| `system_prompt` | text | Added to the project type's instructions for this intent. |
| `answer_guidance` | text | Replaces the project type's answer guidance (optional). |
| `history.policy` | `none`, `recent`, `relevant` | No earlier turns; the recent turns; or the recent turns plus older turns the router marks as related. |
| `retrieval.mode` | `search`, `document`, `reuse`, `none` | Search the indexes; read one whole file or function; reuse the evidence of the related earlier turn; or use no evidence. |
| `retrieval.strategy` | `hybrid` | The search pipeline (semantic, lexical, and code graph, fused). |
| `retrieval.include_related_evidence` | `true`, `false` | When searching, also bring back the evidence of related earlier turns. |
| `retrieval.max_tokens` | 1000–200000 | How much of a document `document` mode may read. |
| `shortcuts` | regular expressions | Whole messages that obviously have this intent (such as "hi"); they skip the router model. |
| `tool` | `{"name": ...}` | Run a tool instead of retrieval and the answer model (tutorial 160). |

In [ ]:
from qachat.project_type import project_type

for intent_id, intent in project_type("literature").intents.items():
    action = (
        f"tool {intent.tool.name}"
        if intent.tool
        else f"{intent.retrieval_mode} evidence"
    )
    print(f"{intent_id:24} history {intent.history_policy:8} {action}")

## 5. How the router uses the intents

The router is a small model call. To see exactly what it reads, the next
cell gives `route_question` a stand-in model that prints its prompt and
answers like the real router would. The prompt lists every intent ID with
its description, the earlier turns, and the new question; the reply is one
line of JSON.

In [ ]:
from qachat.answering import route_question


class PrintingRouter:
    """A stand-in model: prints the router prompt, replies with JSON."""

    def __init__(self, reply):
        self.reply_text = reply
        self.messages = []
        self.thinking = False
        self.temperature = 0.0

    def restore_history(self, messages):
        self.messages = list(messages)

    def configure_generation(self, settings):
        pass

    def reply(self, message, *, max_new_tokens=None):
        # The start (the intents) and the end (the question and format).
        print(message[:600], "\n[...]\n", message[-700:], "\n")
        return self.reply_text


router = PrintingRouter('{"intent": "document_reading", "standalone": true}')
decision = route_question(
    router,
    project_type("literature"),
    [],
    "Read the Smith paper in full and tell me its limitations",
)
print(decision.intent, "| standalone:", decision.standalone)

Shortcuts are checked first. A plain greeting never reaches the model:

In [ ]:
decision = route_question(
    router, project_type("literature"), [], "Hello there!"
)
print(decision.intent, "| shortcut:", decision.shortcut)

## 6. Create your own project type

Suppose you keep meeting notes as Markdown files and want QAChat to answer
questions about them: what was decided, who agreed to do what, and what a
whole meeting covered. A new project type needs no Python, only a JSON
file. This one has five intents, each using a different combination of
history and retrieval:

- `question_answering`: search the notes (the default);
- `decision_log`: search, with an instruction to list decisions and dates;
- `meeting_reading`: read one whole meeting file;
- `conversation_followup`: reuse the evidence of an earlier answer;
- `small_talk`: no evidence, with shortcuts for greetings.

In [ ]:
meeting_notes = {
    "display_name": "Meeting notes",
    "description": "Meeting notes written as Markdown or text files.",
    "extensions": [".md", ".txt"],
    "max_file_bytes": 2_000_000,
    "retrieval_instruction": (
        "Given a question about meetings, retrieve the notes that "
        "answer it"
    ),
    "system_prompt": (
        "You answer questions about the team's meeting notes. Use only "
        "the notes you are given and cite them like [1]. Name the "
        "meeting date when it matters."
    ),
    "capabilities": {"code_graph": False},
    "history": {"recent_turns": 2},
    "default_intent": "question_answering",
    "suggestions": [
        "What did we decide about …?",
        "Who agreed to do what last week?",
        "Read the meeting of … in full",
    ],
    "intents": {
        "question_answering": {
            "description": (
                "A question answered from the notes ('when is the "
                "launch?', 'what did Ana say about the budget?')."
            ),
            "system_prompt": "Answer from the cited notes.",
            "history": {"policy": "relevant"},
            "retrieval": {"mode": "search", "strategy": "hybrid"},
        },
        "decision_log": {
            "description": (
                "A request for decisions or action items: what was "
                "decided, agreed, or assigned, and to whom ('what did "
                "we decide about hiring?', 'list my action items')."
            ),
            "system_prompt": (
                "List each decision or action item on its own line with "
                "the meeting date and the people involved."
            ),
            "history": {"policy": "none"},
            "retrieval": {"mode": "search"},
        },
        "meeting_reading": {
            "description": (
                "A request about one whole meeting, read in full ('go "
                "through the 12 May meeting', 'summarize the whole "
                "kickoff notes'). Not for a question about one detail."
            ),
            "system_prompt": "Cover the whole meeting in order.",
            "history": {"policy": "none"},
            "retrieval": {"mode": "document"},
        },
        "conversation_followup": {
            "description": (
                "A follow-up about an earlier answer that its evidence "
                "can answer ('why?', 'who said that?')."
            ),
            "system_prompt": "Answer from the earlier evidence.",
            "history": {"policy": "relevant"},
            "retrieval": {"mode": "reuse"},
        },
        "small_talk": {
            "description": "Greetings and thanks, not about the notes.",
            "system_prompt": "Reply briefly and offer help.",
            "history": {"policy": "none"},
            "retrieval": {"mode": "none"},
            "shortcuts": ["(hi|hello|hey|thanks|thank you)( qachat)?"],
        },
    },
}

Save it as `meeting_notes.json` in a folder and load that folder. The file
name becomes the type's ID. Everything is validated when it loads.

In [ ]:
folder = root / "sandbox" / "tutorial_150_project_types"
folder.mkdir(parents=True, exist_ok=True)
path = folder / "meeting_notes.json"
path.write_text(json.dumps(meeting_notes, indent=2), encoding="utf-8")

custom = load_project_types(folder)["meeting_notes"]
print(custom.display_name, "|", ", ".join(custom.intents))

A mistake is reported with the type, the intent, and the field, so it is
quick to fix. Here one intent asks for a retrieval mode that does not exist:

In [ ]:
broken = json.loads(json.dumps(meeting_notes))
broken["intents"]["decision_log"]["retrieval"]["mode"] = "everything"
broken_folder = folder / "broken"
broken_folder.mkdir(exist_ok=True)
(broken_folder / "meeting_notes.json").write_text(json.dumps(broken))
try:
    load_project_types(broken_folder)
except ValueError as error:
    print(error)

## 7. Route questions with the real router

Now the real router model classifies questions against the new project
type. It reads only the descriptions, so this is the quickest way to check
that the descriptions separate the intents well. The first run downloads
the model if it is not cached.

In [ ]:
backend = qachat.choose_backend(qachat.BACKEND_AUTO)
model_id = (
    qachat.DEFAULT_CHAT_MODEL_MLX
    if backend == "mlx"
    else qachat.DEFAULT_CHAT_MODEL_TORCH
)
router_model = qachat.ChatSession(
    model_id,
    backend=backend,
    thinking=False,
    temperature=0.0,
    allow_download=True,
)

In [ ]:
questions = [
    "When is the product launch?",
    "What did we decide about hiring a designer?",
    "Go through the whole 12 May meeting",
    "thanks",
]
for question in questions:
    decision = route_question(router_model, custom, [], question)
    print(f"{decision.intent:20} {question}")

If a question lands in the wrong intent, improve the descriptions: add an
example question to the right intent, or say in the wrong one what does not
belong there ("Not for a question about one detail").

## 8. Use it in QAChat

Copy the file into `qachat/project_types/` and restart QAChat. The new type
appears under *Project type* when you create a project, with its
suggestions in the empty chat. The built-in `code` and `literature` types
must stay in that folder.

In [ ]:
installed = PROJECT_TYPES_DIR / "meeting_notes.json"
print(f"To install, copy\n  {path}\nto\n  {installed}")

Next, tutorial 160 shows how an intent runs a **tool** instead of
retrieval and the answer model, using QAChat's research briefing tool as
the example.